In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
openai_client = OpenAI()

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

It depends on the course’s enrollment policy and whether it’s still open.

Usually, you can join now if:
- registration is still open,
- the course hasn’t started yet, or
- the instructor allows late enrollment.

If the course has already started, you may still be able to join if there’s an add/drop period or if the instructor approves it.

If you want, I can help you draft a quick message to the instructor or registrar asking if late enrollment is possible.


In [9]:
context="""
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

https://datatalksclub.github.io/faq/json/llm-zoomcamp.json

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [29]:
documents[1000]

{'id': '641aedbbf5',
 'course': 'llm-zoomcamp',
 'section': 'Module 5: Monitoring',
 'question': 'How set Pandas to show entire text content in a column. Useful to view the entire Explanation column content in the LLM-as-judge section of the offline-rag-evaluation notebook',
 'answer': 'By default, Pandas truncates text content in a column to 50 characters. To view the entire explanation provided by the judge LLM for a non-relevant answer, use the following instruction:\n\n```python\npd.set_option(\'display.max_colwidth\', None)\n```\n\n- **Option:** `display.max_colwidth`\n- **Type:** `int` or `None`\n- **Description:** Sets the maximum width in characters of a column in the representation of a pandas data structure. When a column overflows, a "..." placeholder is used in the output. Setting it to \'None\' allows unlimited width.\n- **Default:** 50\n\nRefer to the [official documentation](https://pandas.pydata.org/docs/user_guide/options.html) for more details.\n\n<{IMAGE:image_1}>'}

In [30]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [39]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

Wrapping it in a function

In [40]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [41]:
question = "I just discovered the course. Can I join now?"
search_results = search(question)

In [42]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [43]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

When we build AI systems, we usually split the prompt into two parts:

- Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
- User prompt: this changes with every request. It carries the actual question and the retrieved context.
We split them because the instructions are fixed and the user prompt is not. Keeping them apart makes the fixed part easy to reuse and the changing part easy to build fresh each time.



In [49]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

The user prompt template has placeholders for the question and the context:

In [51]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

The context is a formatted string with all the search results:

In [45]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [48]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retri

In [52]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

'\nQuestion:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nModule 3: Orchestration\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and expe

Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM.

Now we combine the question with the context into the user prompt:

In [53]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [54]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

The prompt is the bridge between search and the LLM. A bad prompt lets the LLM ignore the context and hallucinate. A good prompt keeps the answer grounded.

Prompt engineering is part art, part science. You experiment, try different things, and see what works. Later in the course we cover evaluation metrics so you can measure how well your prompt performs instead of guessing. For now, this template is a good starting point.